In [1]:
import re
from pathlib import Path

import pandas as pd
from scipy.stats import ttest_ind

In [2]:
version = "eval_v2"
clfs = {"attn"}

paths = sorted(Path("output/input_space_v3").rglob(f"*/{version}/*/eval_table.csv"))
paths = [p for p in paths if p.parts[-2].split("__")[-1] in clfs]
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"(schaefer400|mni_cortex|flat)_lr([-0-9e]+)_([0-9])", key)
    space, lr, run = match.groups()
    table.insert(0, "run", int(run))
    table.insert(0, "base_lr", float(lr))
    table.insert(0, "space", space)
    table.insert(0, "key", key)
    tables.append(table)
state_table = pd.concat(tables, ignore_index=True)
print(state_table.shape)
state_table.head()

48
(168, 20)


,key,space,base_lr,run,model,repr,clf,dataset,ckpt,epoch,lr,wd,hparam_id,hparam,split,loss,acc,acc_std,f1,f1_std
0,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,best,6,0.00069,0.05,29,"[2.3, 1.0]",train,0.009245,0.997842,0.000324,0.997778,0.000360
1,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,best,6,0.00069,0.05,29,"[2.3, 1.0]",validation,0.023990,0.992808,0.001334,0.991433,0.001751
2,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,hcpya_task21,best,6,0.00069,0.05,29,"[2.3, 1.0]",test,0.034005,0.989881,0.001306,0.987513,0.001780
3,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,nsd_cococlip,best,5,0.00036,0.05,25,"[1.2, 1.0]",train,2.099829,0.362703,0.002271,0.297674,0.002460
4,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,attn,nsd_cococlip,best,5,0.00036,0.05,25,"[1.2, 1.0]",validation,2.355878,0.290329,0.005529,0.220301,0.005243


In [3]:
state_summary = state_table.loc[state_table["split"] == "test"].pivot_table(
    values=["acc", "acc_std"], index=["space", "base_lr", "run", "repr", "clf"], columns="dataset"
)
state_summary = state_summary.loc[(slice(None), slice(None), slice(None), "patch", "attn")].dropna(
    axis=1, how="all"
)
state_summary

acc                   acc_std             
dataset                 hcpya_task21 nsd_cococlip hcpya_task21 nsd_cococlip
space       base_lr run                                                    
flat        0.0010  1       0.989881     0.305751     0.001306     0.005450
                    2       0.989683     0.311317     0.001430     0.005401
                    3       0.987302     0.305380     0.001505     0.005364
                    4       0.988095     0.298701     0.001417     0.005308
                    5       0.989881     0.306122     0.001379     0.005498
                    6       0.989484     0.320223     0.001400     0.005467
                    7       0.988690     0.312245     0.001492     0.005658
                    8       0.987698     0.319852     0.001532     0.005570
mni_cortex  0.0010  1       0.961111     0.267532     0.002775     0.005208
                    2       0.960317     0.272727     0.002706     0.005334
                    3       0.957540     0.278850     0.002787     0.005225
                    4       0.963690     0.281076     0.002721     0.005135
                    5       0.959325     0.286827     0.002843     0.005397
                    6       0.966468     0.267347     0.002597     0.005268
                    7       0.965079     0.284045     0.002438     0.005387
                    8       0.965675     0.276994     0.002378     0.005388
schaefer400 0.0003  1       0.977381     0.272542     0.002067     0.005168
                    2       0.974603     0.264750     0.002228     0.005081
                    3       0.975000     0.281262     0.002203     0.005453
                    4       0.976190     0.279592     0.002228     0.005624
                    5       0.974008     0.273469     0.002306     0.005096
                    6       0.975198     0.273098     0.002256     0.004823
                    7       0.969643     0.273840     0.002458     0.005050
                    8       0.974802     0.279963     0.002222     0.005613

In [4]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    "abide_age": "ABIDE Age",
    "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    "ppmi_sex": "PPMI Sex",
    "ppmi_age": "PPMI Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    "hcpya_rest1lr_age": "HCP-YA Age",
    "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}


def format_acc_std(acc, std):
    """Format accuracy and std as 'XX.XX ± YY.YY'"""
    if pd.isna(acc) or pd.isna(std):
        return "—"
    return f"{acc * 100:.1f} ± {std * 100:.1f}"
    # return f"{acc * 100:.1f} \mypm{{{std * 100:.1f}}}"

In [5]:
SPACE_NAMES = {
    "schaefer400": "parcel",
    "flat": "flat",
    "mni_cortex": "volume",
}

SPACE_LRS = {
    "schaefer400": 3e-4,
    "flat": 1e-3,
    "mni_cortex": 1e-3,
}

In [6]:
state_datasets = ["hcpya_task21", "nsd_cococlip"]

records = []

for (space, base_lr, run), row in state_summary.iterrows():
    if base_lr != SPACE_LRS[space]:
        continue
    record = {"space": space, "lr": base_lr, "run": run}
    for ds in state_datasets:
        acc = row[("acc", ds)]
        std = row[("acc_std", ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "\mypm"))

| space       |     lr |   run | HCP-YA Task21   | NSD COCO24   |
|:------------|-------:|------:|:----------------|:-------------|
| flat        | 0.001  |     1 | 99.0 ± 0.1      | 30.6 ± 0.5   |
| flat        | 0.001  |     2 | 99.0 ± 0.1      | 31.1 ± 0.5   |
| flat        | 0.001  |     3 | 98.7 ± 0.2      | 30.5 ± 0.5   |
| flat        | 0.001  |     4 | 98.8 ± 0.1      | 29.9 ± 0.5   |
| flat        | 0.001  |     5 | 99.0 ± 0.1      | 30.6 ± 0.5   |
| flat        | 0.001  |     6 | 98.9 ± 0.1      | 32.0 ± 0.5   |
| flat        | 0.001  |     7 | 98.9 ± 0.1      | 31.2 ± 0.6   |
| flat        | 0.001  |     8 | 98.8 ± 0.2      | 32.0 ± 0.6   |
| mni_cortex  | 0.001  |     1 | 96.1 ± 0.3      | 26.8 ± 0.5   |
| mni_cortex  | 0.001  |     2 | 96.0 ± 0.3      | 27.3 ± 0.5   |
| mni_cortex  | 0.001  |     3 | 95.8 ± 0.3      | 27.9 ± 0.5   |
| mni_cortex  | 0.001  |     4 | 96.4 ± 0.3      | 28.1 ± 0.5   |
| mni_cortex  | 0.001  |     5 | 95.9 ± 0.3      | 28.7 ± 0.5   |
| mni_cort

In [7]:
records = []
for space, name in SPACE_NAMES.items():
    record = {"space": name}
    for ds in state_datasets:
        acc = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].mean()
        std = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "\mypm"))

| space   | HCP-YA Task21   | NSD COCO24   |
|:--------|:----------------|:-------------|
| parcel  | 97.5 ± 0.2      | 27.5 ± 0.5   |
| flat    | 98.9 ± 0.1      | 31.0 ± 0.7   |
| volume  | 96.2 ± 0.3      | 27.7 ± 0.7   |
\begin{tabular}{lll}
\toprule
space & HCP-YA Task21 & NSD COCO24 \\
\midrule
parcel & 97.5 \mypm 0.2 & 27.5 \mypm 0.5 \\
flat & 98.9 \mypm 0.1 & 31.0 \mypm 0.7 \\
volume & 96.2 \mypm 0.3 & 27.7 \mypm 0.7 \\
\bottomrule
\end{tabular}



In [8]:
spaces = ["schaefer400", "flat", "mni_cortex"]

test_results = {}

for ds in state_datasets:
    for ii in range(2):
        for jj in range(ii + 1, 3):
            space_a = spaces[ii]
            space_b = spaces[jj]
            acc_a = state_summary.loc[(space_a, SPACE_LRS[space_a]), ("acc", ds)].values
            acc_b = state_summary.loc[(space_b, SPACE_LRS[space_b]), ("acc", ds)].values
            result = ttest_ind(acc_a, acc_b, equal_var=False)
            result_fmt = f"t={result.statistic:.1f} p={result.pvalue:.1e}"
            test_results[space_a, space_b, ds] = {
                "statistic": result.statistic,
                "pvalue": result.pvalue,
            }
            print(ds, space_a, space_b, result_fmt)

hcpya_task21 schaefer400 flat t=-16.2 p=2.1e-08
hcpya_task21 schaefer400 mni_cortex t=8.7 p=1.3e-06
hcpya_task21 flat mni_cortex t=21.8 p=1.1e-08
nsd_cococlip schaefer400 flat t=-10.8 p=9.0e-08
nsd_cococlip schaefer400 mni_cortex t=-0.7 p=5.2e-01
nsd_cococlip flat mni_cortex t=9.0 p=3.5e-07


In [9]:
version = "eval_v2"
clfs = {"logistic"}

paths = sorted(Path("output/input_space_v3").rglob(f"*/{version}/*/eval_table.csv"))
paths = [p for p in paths if p.parts[-2].split("__")[-1] in clfs]
print(len(paths))

tables = []
for path in paths:
    table = pd.read_csv(path)
    key = path.parts[-4]
    match = re.search(r"(schaefer400|mni_cortex|flat)_lr([-0-9e]+)_([0-9])", key)
    space, lr, run = match.groups()
    table.insert(0, "run", int(run))
    table.insert(0, "base_lr", float(lr))
    table.insert(0, "space", space)
    table.insert(0, "key", key)
    tables.append(table)
trait_table = pd.concat(tables, ignore_index=True)
print(trait_table.shape)
trait_table.head()

144
(29088, 17)


,key,space,base_lr,run,model,repr,clf,dataset,trial,C,split,acc,acc_std,f1,f1_std,bacc,bacc_std
0,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,logistic,aabc_age,NaN,21.544347,train,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000
1,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,logistic,aabc_age,NaN,21.544347,test,0.365385,0.060363,0.361345,0.059764,0.357830,0.060086
2,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,logistic,aabc_age,1.0,0.046416,train,0.842520,0.016223,0.842537,0.016317,0.843452,0.016163
3,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,logistic,aabc_age,1.0,0.046416,test,0.519231,0.063822,0.516382,0.063358,0.518773,0.063582
4,flat_lr1e-3_1,flat,0.001,1,flat_mae,patch,logistic,aabc_age,2.0,0.000774,train,0.572835,0.020757,0.566319,0.021415,0.573496,0.020793


In [10]:
def sem(x):
    return x.std() / (len(x) ** 0.5)


trait_repr = "patch"

trait_summary = trait_table.query("split == 'test' and trial > 0").pivot_table(
    values=["acc", "bacc"],
    index=["space", "base_lr", "run", "repr", "clf"],
    columns="dataset",
    aggfunc=["mean", sem],
)
trait_summary = trait_summary.loc[
    (slice(None), slice(None), slice(None), trait_repr, "logistic")
].dropna(axis=1, how="all")
trait_summary

mean                                 \
                              acc                                  
dataset                  aabc_age  aabc_sex  abide_dx adhd200_dx   
space       base_lr run                                            
flat        0.0010  1    0.475192  0.884727  0.634597   0.605231   
                    2    0.491538  0.884000  0.600484   0.599692   
                    3    0.458654  0.889818  0.632097   0.608000   
                    4    0.476346  0.876000  0.619194   0.629077   
                    5    0.483654  0.881455  0.624516   0.612923   
                    6    0.458462  0.870909  0.611532   0.596769   
                    7    0.468269  0.877455  0.630081   0.599846   
                    8    0.502308  0.873636  0.609677   0.605231   
mni_cortex  0.0010  1    0.536981  0.870862  0.610242   0.595538   
                    2    0.537736  0.877586  0.622903   0.599538   
                    3    0.533962  0.876034  0.610000   0.593077   
                    4    0.529811  0.874310  0.604839   0.597385   
                    5    0.530000  0.867759  0.613306   0.598769   
                    6    0.535094  0.859655  0.610484   0.599385   
                    7    0.530000  0.862586  0.618548   0.625538   
                    8    0.520943  0.864483  0.598306   0.598462   
schaefer400 0.0003  1    0.439808  0.711818  0.613871   0.585846   
                    2    0.442115  0.732364  0.636613   0.592769   
                    3    0.438654  0.718182  0.625726   0.589846   
                    4    0.449423  0.711818  0.617823   0.591231   
                    5    0.452692  0.735818  0.635887   0.574308   
                    6    0.446346  0.738182  0.624919   0.585692   
                    7    0.442308  0.718364  0.625806   0.591077   
                    8    0.439423  0.722727  0.629516   0.590923   

                                                                             \
                                                   bacc                       
dataset                 adni_ad_vs_cn ppmi_dx  aabc_age  aabc_sex  abide_dx   
space       base_lr run                                                       
flat        0.0010  1        0.724878  0.6252  0.474332  0.878804  0.629764   
                    2        0.725854  0.6312  0.490598  0.878546  0.593965   
                    3        0.722927  0.6470  0.456536  0.886726  0.627295   
                    4        0.744390  0.6117  0.474858  0.870265  0.612994   
                    5        0.730244  0.6209  0.482443  0.875931  0.618713   
                    6        0.746098  0.6228  0.456671  0.865768  0.605236   
                    7        0.752439  0.6359  0.466815  0.871637  0.624512   
                    8        0.737317  0.6158  0.500760  0.868111  0.601355   
mni_cortex  0.0010  1        0.753659  0.6467  0.539382  0.864363  0.604785   
                    2        0.754146  0.6244  0.539670  0.872672  0.615982   
                    3        0.737561  0.6332  0.536044  0.870735  0.602012   
                    4        0.770488  0.6171  0.532170  0.869081  0.597463   
                    5        0.751463  0.6352  0.532404  0.862083  0.605751   
                    6        0.756341  0.6220  0.537569  0.852782  0.603902   
                    7        0.767805  0.6194  0.532239  0.856691  0.611759   
                    8        0.743902  0.6381  0.523228  0.858738  0.592012   
schaefer400 0.0003  1        0.727561  0.6518  0.437985  0.702024  0.607700   
                    2        0.730976  0.6368  0.440080  0.721332  0.631334   
                    3        0.735122  0.6646  0.437122  0.704436  0.619958   
                    4        0.749268  0.6604  0.448004  0.699090  0.611444   
                    5        0.736585  0.6486  0.451763  0.724789  0.631035   
                    6        0.724878  0.6331  0.444206  0.724558  0.619317   
                    7        0.740976  0.6464  0.440041  0.705815  0.6

In [11]:
trait_datasets = ["abide_dx", "adhd200_dx", "adni_ad_vs_cn", "ppmi_dx", "aabc_age", "aabc_sex"]
trait_metric = "bacc"

records = []

for (space, base_lr, run), row in trait_summary.iterrows():
    if base_lr != SPACE_LRS[space]:
        continue
    record = {"space": space, "lr": base_lr, "run": run}
    for ds in trait_datasets:
        acc = row[("mean", trait_metric, ds)]
        std = row[("sem", trait_metric, ds)]
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
print(result_fmt.to_latex(index=False).replace("±", "\mypm"))

| space       |     lr |   run | ABIDE Dx   | ADHD200 Dx   | ADNI Dx    | PPMI Dx    | HCP-A Age   | HCP-A Sex   |
|:------------|-------:|------:|:-----------|:-------------|:-----------|:-----------|:------------|:------------|
| flat        | 0.001  |     1 | 63.0 ± 0.4 | 58.9 ± 0.5   | 61.3 ± 0.8 | 58.9 ± 0.4 | 47.4 ± 0.6  | 87.9 ± 0.4  |
| flat        | 0.001  |     2 | 59.4 ± 0.4 | 58.4 ± 0.6   | 61.3 ± 0.8 | 58.9 ± 0.4 | 49.1 ± 0.6  | 87.9 ± 0.5  |
| flat        | 0.001  |     3 | 62.7 ± 0.4 | 59.2 ± 0.6   | 60.5 ± 0.8 | 60.7 ± 0.4 | 45.7 ± 0.5  | 88.7 ± 0.5  |
| flat        | 0.001  |     4 | 61.3 ± 0.4 | 61.4 ± 0.5   | 64.3 ± 0.8 | 57.8 ± 0.5 | 47.5 ± 0.6  | 87.0 ± 0.4  |
| flat        | 0.001  |     5 | 61.9 ± 0.4 | 59.7 ± 0.6   | 62.4 ± 0.9 | 58.4 ± 0.5 | 48.2 ± 0.7  | 87.6 ± 0.4  |
| flat        | 0.001  |     6 | 60.5 ± 0.4 | 58.2 ± 0.5   | 62.5 ± 0.8 | 58.2 ± 0.5 | 45.7 ± 0.6  | 86.6 ± 0.5  |
| flat        | 0.001  |     7 | 62.5 ± 0.4 | 58.7 ± 0.6   | 64.4 ± 0.8 | 59.8 ±

In [12]:
for ds in trait_datasets:
    for ii in range(2):
        for jj in range(ii + 1, 3):
            space_a = spaces[ii]
            space_b = spaces[jj]
            acc_a = trait_summary.loc[
                (space_a, SPACE_LRS[space_a]), ("mean", trait_metric, ds)
            ].values
            acc_b = trait_summary.loc[
                (space_b, SPACE_LRS[space_b]), ("mean", trait_metric, ds)
            ].values
            result = ttest_ind(acc_a, acc_b, equal_var=False)
            result_fmt = f"t={result.statistic:.1f} p={result.pvalue:.1e}"
            test_results[space_a, space_b, ds] = {
                "statistic": result.statistic,
                "pvalue": result.pvalue,
            }
            print(ds, space_a, space_b, result_fmt)

abide_dx schaefer400 flat t=1.1 p=2.8e-01
abide_dx schaefer400 mni_cortex t=4.1 p=1.1e-03
abide_dx flat mni_cortex t=1.9 p=8.7e-02
adhd200_dx schaefer400 flat t=-5.8 p=1.1e-04
adhd200_dx schaefer400 mni_cortex t=-4.5 p=9.0e-04
adhd200_dx flat mni_cortex t=0.8 p=4.4e-01
adni_ad_vs_cn schaefer400 flat t=-1.3 p=2.3e-01
adni_ad_vs_cn schaefer400 mni_cortex t=-3.9 p=2.0e-03
adni_ad_vs_cn flat mni_cortex t=-2.6 p=2.3e-02
ppmi_dx schaefer400 flat t=4.5 p=5.1e-04
ppmi_dx schaefer400 mni_cortex t=3.8 p=2.1e-03
ppmi_dx flat mni_cortex t=-0.5 p=6.2e-01
aabc_age schaefer400 flat t=-5.7 p=3.5e-04
aabc_age schaefer400 mni_cortex t=-34.3 p=6.6e-15
aabc_age flat mni_cortex t=-10.1 p=4.6e-06
aabc_sex schaefer400 flat t=-36.7 p=9.5e-14
aabc_sex schaefer400 mni_cortex t=-33.8 p=1.6e-13
aabc_sex flat mni_cortex t=3.2 p=6.8e-03


In [13]:
records = []

for space in ["schaefer400", "mni_cortex", "flat"]:
    record = {"space": SPACE_NAMES[space]}
    for ds in trait_datasets:
        acc = trait_summary.loc[(space, SPACE_LRS[space]), ("mean", trait_metric, ds)].mean()
        std = trait_summary.loc[(space, SPACE_LRS[space]), ("mean", trait_metric, ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    for ds in state_datasets:
        acc = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].mean()
        std = state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].std()
        record[DATASET_NAMES.get(ds, ds)] = format_acc_std(acc, std)
    records.append(record)

result_fmt = pd.DataFrame(records)
print(result_fmt.to_markdown(index=False))
# print(result_fmt.to_latex(index=False).replace("±", "\mypm"))
print(re.sub("± ([.0-9]+)", r"\\mypm{\1}", result_fmt.to_latex(index=False)))

| space   | ABIDE Dx   | ADHD200 Dx   | ADNI Dx    | PPMI Dx    | HCP-A Age   | HCP-A Sex   | HCP-YA Task21   | NSD COCO24   |
|:--------|:-----------|:-------------|:-----------|:-----------|:------------|:------------|:----------------|:-------------|
| parcel  | 62.0 ± 0.8 | 56.8 ± 0.6   | 61.6 ± 1.2 | 61.4 ± 1.3 | 44.2 ± 0.5  | 71.2 ± 1.0  | 97.5 ± 0.2      | 27.5 ± 0.5   |
| volume  | 60.4 ± 0.8 | 58.8 ± 1.1   | 64.3 ± 1.6 | 59.1 ± 1.2 | 53.4 ± 0.5  | 86.3 ± 0.7  | 96.2 ± 0.3      | 27.7 ± 0.7   |
| flat    | 61.4 ± 1.3 | 59.2 ± 1.0   | 62.4 ± 1.4 | 58.8 ± 1.1 | 47.5 ± 1.6  | 87.4 ± 0.7  | 98.9 ± 0.1      | 31.0 ± 0.7   |
\begin{tabular}{lllllllll}
\toprule
space & ABIDE Dx & ADHD200 Dx & ADNI Dx & PPMI Dx & HCP-A Age & HCP-A Sex & HCP-YA Task21 & NSD COCO24 \\
\midrule
parcel & 62.0 \mypm{0.8} & 56.8 \mypm{0.6} & 61.6 \mypm{1.2} & 61.4 \mypm{1.3} & 44.2 \mypm{0.5} & 71.2 \mypm{1.0} & 97.5 \mypm{0.2} & 27.5 \mypm{0.5} \\
volume & 60.4 \mypm{0.8} & 58.8 \mypm{1.1} & 64.3 \mypm{1.6}

In [14]:
test_table = pd.DataFrame.from_dict(test_results, orient="index")
test_table

statistic        pvalue
schaefer400 flat       hcpya_task21  -16.192664  2.066093e-08
            mni_cortex hcpya_task21    8.669592  1.271812e-06
flat        mni_cortex hcpya_task21   21.751867  1.127196e-08
schaefer400 flat       nsd_cococlip  -10.792715  8.988062e-08
            mni_cortex nsd_cococlip   -0.661291  5.200377e-01
flat        mni_cortex nsd_cococlip    8.978147  3.514693e-07
schaefer400 flat       abide_dx        1.133011  2.795663e-01
            mni_cortex abide_dx        4.087776  1.129752e-03
flat        mni_cortex abide_dx        1.878184  8.665007e-02
schaefer400 flat       adhd200_dx     -5.780409  1.061844e-04
            mni_cortex adhd200_dx     -4.511173  9.021868e-04
flat        mni_cortex adhd200_dx      0.797363  4.386287e-01
schaefer400 flat       adni_ad_vs_cn  -1.262196  2.278436e-01
            mni_cortex adni_ad_vs_cn  -3.856789  1.971104e-03
flat        mni_cortex adni_ad_vs_cn  -2.564917  2.275062e-02
schaefer400 flat       ppmi_dx         4.519677  5.122529e-04
            mni_cortex ppmi_dx         3.765445  2.090329e-03
flat        mni_cortex ppmi_dx        -0.511083  6.174129e-01
schaefer400 flat       aabc_age       -5.697497  3.465110e-04
            mni_cortex aabc_age      -34.274838  6.625161e-15
flat        mni_cortex aabc_age      -10.057261  4.622377e-06
schaefer400 flat       aabc_sex      -36.709742  9.453851e-14
            mni_cortex aabc_sex      -33.832918  1.589947e-13
flat        mni_cortex aabc_sex        3.170939  6.811832e-03

In [15]:
cutoff = 1e-4
cutoff = float(f"{cutoff:.0e}")
print(cutoff)
test_table.query(f"pvalue < {cutoff}")

0.0001


statistic        pvalue
schaefer400 flat       hcpya_task21 -16.192664  2.066093e-08
            mni_cortex hcpya_task21   8.669592  1.271812e-06
flat        mni_cortex hcpya_task21  21.751867  1.127196e-08
schaefer400 flat       nsd_cococlip -10.792715  8.988062e-08
flat        mni_cortex nsd_cococlip   8.978147  3.514693e-07
schaefer400 mni_cortex aabc_age     -34.274838  6.625161e-15
flat        mni_cortex aabc_age     -10.057261  4.622377e-06
schaefer400 flat       aabc_sex     -36.709742  9.453851e-14
            mni_cortex aabc_sex     -33.832918  1.589947e-13

In [16]:
cutoff = 0.05 / len(test_table)
cutoff = float(f"{cutoff:.0e}")
print(cutoff)
test_table.query(f"pvalue < {cutoff}")

0.002


statistic        pvalue
schaefer400 flat       hcpya_task21  -16.192664  2.066093e-08
            mni_cortex hcpya_task21    8.669592  1.271812e-06
flat        mni_cortex hcpya_task21   21.751867  1.127196e-08
schaefer400 flat       nsd_cococlip  -10.792715  8.988062e-08
flat        mni_cortex nsd_cococlip    8.978147  3.514693e-07
schaefer400 mni_cortex abide_dx        4.087776  1.129752e-03
            flat       adhd200_dx     -5.780409  1.061844e-04
            mni_cortex adhd200_dx     -4.511173  9.021868e-04
                       adni_ad_vs_cn  -3.856789  1.971104e-03
            flat       ppmi_dx         4.519677  5.122529e-04
                       aabc_age       -5.697497  3.465110e-04
            mni_cortex aabc_age      -34.274838  6.625161e-15
flat        mni_cortex aabc_age      -10.057261  4.622377e-06
schaefer400 flat       aabc_sex      -36.709742  9.453851e-14
            mni_cortex aabc_sex      -33.832918  1.589947e-13

In [17]:
import json

sigmas = {}

space = "flat"
for ds in trait_datasets:
    sigmas[ds] = round(
        trait_summary.loc[(space, SPACE_LRS[space]), ("mean", trait_metric, ds)].std(), 4
    )
for ds in state_datasets:
    sigmas[ds] = round(state_summary.loc[(space, SPACE_LRS[space]), ("acc", ds)].std(), 4)
print(json.dumps(sigmas, indent=4))

{
    "abide_dx": 0.0131,
    "adhd200_dx": 0.0102,
    "adni_ad_vs_cn": 0.014,
    "ppmi_dx": 0.0107,
    "aabc_age": 0.0156,
    "aabc_sex": 0.0069,
    "hcpya_task21": 0.001,
    "nsd_cococlip": 0.0075
}
